In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import PyMCTranslate

sys.path.insert(0, '../data_processing/palette/')
sys.path.insert(0, '../data_processing/util/')

from palette import Palette
from util import array_to_schematic

INFO - PyMCTranslate Version 385


### Load Block Palettes and Dataframes

In [2]:
# Get the embeddings
# embeddings_path = '../text2mc-source-code/block2vec/output/block2vec/embeddings.json'
embeddings_path = './embeddings_newline.json'

java_palette_path = './java_palette_newline.json'
block_dist_path = 'block_dist_dataframe.csv'

with open(embeddings_path) as f:
    embeddings = json.load(f)

# with open('./embeddings_newline.json', "w") as f:
#     json.dump(embeddings, f, indent=4)

with open(java_palette_path) as f:
    java_palette_block2token = json.load(f)

block_dist_df = pd.read_csv(block_dist_path)

### Create Palette Objects

In [3]:
embed_blocks = list(embeddings.keys())
embed_blocks.remove('UNKNOWN_BLOCK')
embed_palette = Palette(embed_blocks)

java_palette_strings = list(java_palette_block2token.keys())
java_palette = Palette(java_palette_strings)

Added 87 blocks for potential tranformations
Added 556 blocks for potential tranformations


In [4]:
threshold = block_dist_df["total_occurences"].quantile(0.75)
mask_blocks = block_dist_df[block_dist_df["total_occurences"] < threshold]
mask_list = mask_blocks["Block"].to_list()

In [5]:
num_masked_blocks = mask_blocks["total_occurences"].sum()
num_total_blocks = block_dist_df["total_occurences"].sum()
print(num_masked_blocks)
print(num_total_blocks)

6363547
9547743615


In [6]:
java_mask_arr = java_palette.get_mask_lookup(mask_list)
java_unmasked_blocks = list(set([java_palette.token2block[i] for i in java_mask_arr]))

In [7]:
len(java_palette.block_strings)
# len(java_unmasked_blocks)

9498

In [16]:
keep_blockstates = ["axis", "facing", "shape", "east", "west", "north", "south", "face", "half"]
keep_blockstates = ["axis", "facing", "shape"]

java_unmasked_palette = Palette(java_unmasked_blocks)

reduced_embeddings, src2tgt, tgt2src = embed_palette.reduce_blockstates(keep_blockstates)
reduced_java, src2tgt, tgt2src = java_unmasked_palette.reduce_blockstates(keep_blockstates)
block_ids_strip = [
    "minecraft:bed",
    "minecraft:black_shulker_box",
    "minecraft:chest",
    "minecraft:deepslate",
    "minecraft:quartz_pillar",
    "minecraft:deepslate",
    "minecraft:repeater",
    "minecraft:wall_torch",
    "minecraft:rail"
]

reduced_embeddings, src2tgt, tgt2src = reduced_embeddings.reduce_blockstates(keep_blockstates=[], block_ids=block_ids_strip)
reduced_java, src2tgt, tgt2src = reduced_java.reduce_blockstates(keep_blockstates=[], block_ids=block_ids_strip)


Added 0 blocks for potential tranformations
minecraft:acacia_button {'facing': 'south'}
minecraft:acacia_button {'facing': 'west'}
minecraft:acacia_button {'facing': 'south'}
minecraft:acacia_button {'facing': 'west'}
minecraft:acacia_button {'facing': 'north'}
minecraft:acacia_button {'facing': 'east'}
minecraft:acacia_button {'facing': 'south'}
minecraft:acacia_button {'facing': 'west'}
minecraft:acacia_door {'facing': 'north'}
minecraft:acacia_door {'facing': 'east'}
minecraft:acacia_door {'facing': 'south'}
minecraft:acacia_door {'facing': 'west'}
minecraft:acacia_fence None
minecraft:acacia_fence_gate {'facing': 'south'}
minecraft:acacia_fence_gate {'facing': 'west'}
minecraft:acacia_hanging_sign {'facing': 'north'}
minecraft:acacia_hanging_sign {'facing': 'east'}
minecraft:acacia_hanging_sign {'facing': 'south'}
minecraft:acacia_hanging_sign {'facing': 'west'}
minecraft:acacia_leaves None
minecraft:acacia_log {'axis': 'x'}
minecraft:acacia_log {'axis': 'y'}
minecraft:acacia_log {

In [17]:
embed_list = reduced_embeddings.gdpc_blocks
java_list = reduced_java.gdpc_blocks

In [18]:
inter = [x for x in embed_list if any(x == y for y in java_list)]
diff_embed = [str(x) for x in embed_list if not any(x == y for y in java_list)]
diff_java = [str(x) for x in java_list if not any(x == y for y in embed_list)]

In [19]:
diff_java.sort()
for i in diff_java:
    print(i)